# BME 2301 2026 Final Project

# Machine Learning Classification of Melanoma from fusing Skin Lesion Images and Demographic Data

**Kate Li**  
---

## Project Question

> Which machine learning techniques would most accurately distinguish between benign and malignant skin lesions, and how can we incorporate non-image data (patient statistics) improve classification performance? 

### Load and show your dataset(s)

My directory structure: 
melanoma_cancer_dataset/
  train/
    benign/
    malignant/
  test/
    benign/
    malignant/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import random
from PIL import Image

base_path = "melanoma_cancer_dataset/train"

categories = ["benign", "malignant"]

plt.figure(figsize=(10, 6))

plot_index = 1

for category in categories:
    category_path = os.path.join(base_path, category)
    
    # Get all jpg images
    images = [img for img in os.listdir(category_path) if img.endswith(".jpg")]
    
    # Randomly select 3 images
    random_images = random.sample(images, 3)
    
    for img_name in random_images:
        img_path = os.path.join(category_path, img_name)
        img = Image.open(img_path)
        
        plt.subplot(2, 3, plot_index)
        plt.imshow(img)
        plt.title(category)
        plt.axis("off")
        
        plot_index += 1

plt.suptitle("Random Samples from Train Set: Benign vs Malignant")
plt.tight_layout()
plt.show()

In [ ]:
import os

base_path = "melanoma_cancer_dataset"

splits = ["train", "test"]
categories = ["benign", "malignant"]

for split in splits:
    print(f"\n{split.upper()} SET")
    print("-" * 20)
    
    for category in categories:
        folder_path = os.path.join(base_path, split, category)
        
        # Count only .jpg files
        image_count = len([file for file in os.listdir(folder_path) if file.endswith(".jpg")])
        
        print(f"{category.capitalize()}: {image_count} images")

print()
print("after counting the images, I have determined that there is not a class imbalance, so no cleaning needs to be done")

## Exploratory Visualization - Data Preprocessing

With my synthetic patient demographic dataset, I can visualize the age distribution, skin color, and previous treatment and how effective it was in treating melanoma. 

Age is a known risk factor in melanoma. Visualizing its distribution if it is skewed toward a specific age group. Previous treatment and its effectiveness also provide context for clinical progression. If certain treatments are disproportionately associated with poor outcomes, this could influence predictive modeling. Additionally, understanding these distributions helps determine whether treatment history should be included as a feature in the classification model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("askin_disease_dataset.csv")

print(df.head())

print(df.info())

print(df.describe())

print()

In [ ]:
# Filter for Melanoma cases
melanoma_df = df[df["Disease_Type"] == "Melanoma"]

print(melanoma_df.head())

# Basic statistics for Age
print("\nMelanoma Age Statistics:")
print(melanoma_df["Age"].describe())


In [ ]:
import matplotlib.pyplot as plt

melanoma_df = df[df["Disease_Type"] == "Melanoma"]

plt.figure()
plt.hist(melanoma_df["Age"], bins=30)
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.title("Age Distribution of Melanoma Patients")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

male_age = melanoma_df[melanoma_df["Gender"] == "Male"]["Age"]
female_age = melanoma_df[melanoma_df["Gender"] == "Female"]["Age"]

# Create boxplot
plt.figure()
plt.boxplot([male_age, female_age], labels=["Male", "Female"])

plt.ylabel("Age")
plt.title("Age Distribution of Melanoma Patients by Gender")
plt.show()

print("this data is not very interesting because it is synthetically generated to be very balanced")

## Ditch clinical statistics (sadly), Test/compare Machine learning methods

1. CNN
2. SVM
3. Random Forest

In [ ]:
# CNN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Paths
train_dir = "melanoma_cancer_dataset/train"
test_dir = "melanoma_cancer_dataset/test"

# Parameters
img_height = 180
img_width = 180
batch_size = 32

# Load datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',  # binary classification
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='binary',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

# Normalize pixel values (0–255 → 0–1)
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# Build CNN model
model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, 3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  # binary output
])

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
epochs = 10
history = model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds
)

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(test_ds)

print(f"\nTest Accuracy: {test_accuracy:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Get predictions
y_true = []
y_pred = []
images_list = []

for images, labels in test_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype(int).flatten()
    
    y_true.extend(labels.numpy().astype(int).flatten())
    y_pred.extend(preds)
    images_list.extend(images.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
images_list = np.array(images_list)

# Indices
correct_indices = np.where(y_true == y_pred)[0]
false_positive_indices = np.where((y_true == 0) & (y_pred == 1))[0]
false_negative_indices = np.where((y_true == 1) & (y_pred == 0))[0]

# Random selections (with safety checks)
correct_idx = random.choice(correct_indices) if len(correct_indices) > 0 else None
fp_idx = random.choice(false_positive_indices) if len(false_positive_indices) > 0 else None
fn_idx = random.choice(false_negative_indices) if len(false_negative_indices) > 0 else None

class_names = ["benign", "malignant"]

# Plot
plt.figure(figsize=(12,4))

# Correct
if correct_idx is not None:
    plt.subplot(1,3,1)
    plt.imshow(images_list[correct_idx])
    plt.title(f"Correct\nTrue: {class_names[y_true[correct_idx]]}\nPred: {class_names[y_pred[correct_idx]]}")
    plt.axis("off")

# False Positive
if fp_idx is not None:
    plt.subplot(1,3,2)
    plt.imshow(images_list[fp_idx])
    plt.title(f"False Positive\nTrue: benign\nPred: malignant")
    plt.axis("off")

# False Negative
if fn_idx is not None:
    plt.subplot(1,3,3)
    plt.imshow(images_list[fn_idx])
    plt.title(f"False Negative\nTrue: malignant\nPred: benign")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Counts
num_correct = len(correct_indices)
num_false_positive = len(false_positive_indices)
num_false_negative = len(false_negative_indices)

total = len(y_true)

print(f"Total samples: {total}")
print(f"Correct predictions: {num_correct}")
print(f"False Positives (benign → malignant): {num_false_positive}")
print(f"False Negatives (malignant → benign): {num_false_negative}")

# Optional: percentages
print("\nPercentages:")
print(f"False Positive Rate: {num_false_positive / total:.4f}")
print(f"False Negative Rate: {num_false_negative / total:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Indices for each category
correct_benign = np.where((y_true == 0) & (y_pred == 0))[0]
correct_malignant = np.where((y_true == 1) & (y_pred == 1))[0]
false_positive = np.where((y_true == 0) & (y_pred == 1))[0]
false_negative = np.where((y_true == 1) & (y_pred == 0))[0]

# Random selections (safe)
cb_idx = random.choice(correct_benign) if len(correct_benign) > 0 else None
cm_idx = random.choice(correct_malignant) if len(correct_malignant) > 0 else None
fp_idx = random.choice(false_positive) if len(false_positive) > 0 else None
fn_idx = random.choice(false_negative) if len(false_negative) > 0 else None

class_names = ["benign", "malignant"]

# Plot 4 images
plt.figure(figsize=(12,8))

# Correct benign
if cb_idx is not None:
    plt.subplot(2,2,1)
    plt.imshow(images_list[cb_idx])
    plt.title("Correct Benign\nTrue: benign | Pred: benign")
    plt.axis("off")

# Correct malignant
if cm_idx is not None:
    plt.subplot(2,2,2)
    plt.imshow(images_list[cm_idx])
    plt.title("Correct Malignant\nTrue: malignant | Pred: malignant")
    plt.axis("off")

# False positive
if fp_idx is not None:
    plt.subplot(2,2,3)
    plt.imshow(images_list[fp_idx])
    plt.title("False Positive\nTrue: benign | Pred: malignant")
    plt.axis("off")

# False negative
if fn_idx is not None:
    plt.subplot(2,2,4)
    plt.imshow(images_list[fn_idx])
    plt.title("False Negative\nTrue: malignant | Pred: benign")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Random Forest: Load data (same structure, but no normalization layer needed here)
import tensorflow as tf
import numpy as np

train_dir = "melanoma_cancer_dataset/train"
test_dir = "melanoma_cancer_dataset/test"

img_height = 224
img_width = 224
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='binary',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False   # IMPORTANT for alignment
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='binary',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

In [10]:
#Load pretrained CNN for feature extraction

# remove the classification head and keep convolutional features.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # freeze model

C:\Users\JIEYAN~1\AppData\Local\Temp/ipykernel_26376/1473566445.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


In [11]:
def extract_features(dataset):
    features = []
    labels = []
    
    for images, y in dataset:
        # Preprocess for MobileNetV2
        images = tf.keras.applications.mobilenet_v2.preprocess_input(images)
        
        feature_batch = base_model.predict(images)
        feature_batch = feature_batch.reshape(feature_batch.shape[0], -1)
        
        features.append(feature_batch)
        labels.append(y.numpy())
    
    return np.vstack(features), np.concatenate(labels)

X_train, y_train = extract_features(train_ds)
X_test, y_test = extract_features(test_ds)

print("Feature shape:", X_train.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 313ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

C:\Users\Jie Yan\anaconda3\lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(n_jobs=-1, random_state=42)

In [13]:
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy (Random Forest): {accuracy:.4f}")

Test Accuracy (Random Forest): 0.9030


In [14]:
# SVM. Scale features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# apply PCA for speedier
from sklearn.decomposition import PCA

pca = PCA(n_components=150, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Reduced shape:", X_train_pca.shape)

In [ ]:
# faster SVM (linear)
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

svm_model = LinearSVC(
    C=1.0,
    max_iter=5000
)

svm_model.fit(X_train_pca, y_train)

y_pred_svm = svm_model.predict(X_test_pca)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Test Accuracy (Fast SVM): {accuracy_svm:.4f}")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

clf = HistGradientBoostingClassifier()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

## Assumptions and Limitations

1. Limitation: The demographic dataset is synthetic and genererated to be balanced. This matters because an artificially balanced dataset gets rid of any differences between classes. Real-world data are imbalanced, so using this balanced data for the model will overestimate performance. Statistical significance may change, and predictive accuracy could drop substantially. Conclusions drawn from balanced synthetic data may therefore not generalize to clinical populations.

2. Assumption: For the images, training and testing data are independent. This matters because independence ensures realistic testing. If violated, data leakage would inflate performance and misrepresent predictive ability.

3. Assumption: Dataset is representative of the broader population. This matters for the model to be used in the real world. If violated, the model would  perform poorly in underrepresented populations and introduce health disparities.


## Reflection and Next Steps

So far, I have learned that preprocessing and visualizing dataset structure are important to know what you are working with. I had this idea to incorporate demographic metadata paired with each image, but I couldn't even find real demographic data for people with skin cancer (may be protected patient information). This limits my initial ideas and project proposal going forward, but I will learn more in the machine learning unit in this class and hope I can do what I can to make a predictive model to feature extract and classify these lesion images. 

I did not have enough time to follow through with creating an app that also asks a survey for symptoms and embed them with the image that the user takes. That can be a task for BDS capstone!